In [7]:
import glob
import os
import datetime
from astropy.table import Table
filtername = 'F480M'
basepath = '/orange/adamginsburg/jwst/w51/'



In [8]:
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
from jwst.datamodels import dqflags

nircam_short_filters = ['F140M', 'F162M', 'F182M', 'F187N', 'F210M' ]
nircam_long_filters = [ 'F335M',  'F360M','F405N', 'F410M',  'F480M']
miri_filters = ['F560W', 'F770W', 'F1000W',  'F1280W',  'F2100W', ]
def tblname_to_imgname(tblname,  proposal_id='6151'):
    filtername = tblname.split('/')[-1].split('_')[0]
    if filtername.upper() in miri_filters:
        target= 'w51_miri'
    else:        
        target = 'w51'

    nvisits = {'2221': {'brick': 1, 'cloudc': 2},
                '1182': {'brick': 2},
                '6151': {'w51': 1, 'w51_miri': 2}
                }
    field_to_reg_mapping = {'2221': {'001': 'brick', '002': 'cloudc'},
                            '1182': {'004': 'brick'},
                            '6151': {'001': 'w51', '002':'w51_miri'}}[proposal_id]
    reg_to_field_mapping = {v:k for k,v in field_to_reg_mapping.items()}
    field = reg_to_field_mapping[target]


    
    module = tblname.split('/')[-1].split('_')[1]
    visit = tblname.split('/')[-1].split('_')[2]
    visitid = visit[5:]
    vgroup = tblname.split('/')[-1].split('_')[3]
    vgroupid = vgroup[6:]
    exposure = tblname.split('/')[-1].split('_')[4]
    expid = exposure[3:]
    imgname = f'{basepath}/{filtername.upper()}/pipeline/jw0{proposal_id}{field}{visitid}_{vgroupid}_{expid}_{module}_cal.fits'


    return imgname
def load_data(filename):
    fh = fits.open(filename)
    im1 = fh
    data = im1['SCI'].data
    try:
        wht = im1['WHT'].data
    except KeyError:
        wht = None
    err = im1['ERR'].data
    instrument = im1[0].header['INSTRUME']
    telescope = im1[0].header['TELESCOP']
    obsdate = im1[0].header['DATE-OBS']
    return fh, im1, data, wht, err, instrument, telescope, obsdate
def filter_by_nmatch(tbls_merged, tbls, tolerance=0):

    for jj, tbl_merged in enumerate(tbls_merged):
        # convert skycoord of tbl_merged to pixel coordinates in each tbl in tbls
        skycoord_merged = tbl_merged['skycoord']
        for ii, tbl in enumerate(tbls):
            img_name = tblname_to_imgname(tblfns[ii])
            fh, im1, img_data, wht, err, instrument, telescope, obsdate = load_data(img_name)
            wcs = WCS(im1[1].header)
            pixcoord_merged = wcs.world_to_pixel(skycoord_merged)

            dq = im1['DQ'].data
            # check whether pixel coordiantes of tbl_merged fall within the image field of view
         
            is_saturated = dq & dqflags.pixel['SATURATED']
            is_hot = dq & dqflags.pixel['HOT']
            is_dead = dq & dqflags.pixel['DEAD']
            is_really_saturated = is_saturated & ~is_hot & ~is_dead
            is_nan_but_is_really_saturated = np.isnan(img_data) & is_really_saturated

            img_shape = img_data.shape
            finite_coord = np.isfinite(pixcoord_merged[0]) & np.isfinite(pixcoord_merged[1])
            xpix = pixcoord_merged[0].astype(int)
            ypix = pixcoord_merged[1].astype(int)
            in_fov = finite_coord & (pixcoord_merged[0] >= 0) & (pixcoord_merged[0] < img_shape[1]) & (pixcoord_merged[1] >= 0) & (pixcoord_merged[1] < img_shape[0])
            valid_pixel = np.zeros(in_fov.shape, dtype=bool)
            valid_pixel[in_fov] = np.isfinite(img_data[ypix[in_fov], xpix[in_fov]]) | is_really_saturated[ypix[in_fov], xpix[in_fov]]
            in_fov = in_fov & valid_pixel
            in_fov_int = in_fov.astype(int)
            if ii == 0:
                in_fov_all = in_fov_int
            else:
                in_fov_all = in_fov_all + in_fov_int

        nmatch_max = in_fov_all
        nmatch = tbl_merged['nmatch']
        from_sat_cat = tbl_merged['from_sat_catalog']
        from_sat_cat = from_sat_cat.astype(bool)
        keep = (nmatch >= nmatch_max - tolerance) | from_sat_cat
        tbl_merged_cut = tbl_merged[keep]
        print(f"Number of sources in merged table: {len(tbl_merged)}")
        print(f"Number of sources in merged table with nmatch >= {nmatch_max}: {len(tbl_merged_cut)}")  
        print(f"Number of sources in merged table with nmatch >= {nmatch_max - tolerance}: {len(tbl_merged[nmatch >= nmatch_max - tolerance])}")    
        #save the cut table to a new file
        if tolerance == 0:
            label = 'grade_a'
        elif tolerance ==1:
            label = 'grade_b'
        tbl_merged_cut.write(tblfns_merged[jj].replace('.fits', f'_nmatch_cut_{label}.fits'), overwrite=True)
    
            


In [9]:
filternames=  nircam_short_filters + nircam_long_filters
for filtername in filternames:
    if filtername in nircam_short_filters+nircam_long_filters:
        modules = ['nrca', 'nrcb']
    else:
        modules = ['mirimage']
    for module in modules:
        tblfns = glob.glob(f'{basepath}/{filtername.upper()}/*{module}*daophot_combined_with_satstars.fits')
        for tbl in tblfns:
            print(tbl)
            print('last modified:', datetime.datetime.fromtimestamp(os.path.getmtime(tbl)))

        tblfns_merged = glob.glob(f'/orange/adamginsburg/jwst/w51/catalogs/{filtername.lower()}_{module}_indivexp_merged_dao_after_merger_combined_with_satstars.fits')
        print('last modified:', datetime.datetime.fromtimestamp(os.path.getmtime(tblfns_merged[0])))

        tbls = [Table.read(tblfn) for tblfn in tblfns]
        tbls_merged = [Table.read(tblfn) for tblfn in tblfns_merged]


        filter_by_nmatch(tbls_merged, tbls, tolerance=0)
        filter_by_nmatch(tbls_merged, tbls, tolerance=1)

/orange/adamginsburg/jwst/w51//F140M/f140m_nrca3_visit001_vgroup03109_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:37
/orange/adamginsburg/jwst/w51//F140M/f140m_nrca1_visit001_vgroup03109_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:26
/orange/adamginsburg/jwst/w51//F140M/f140m_nrca3_visit001_vgroup03109_exp00004_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:35
/orange/adamginsburg/jwst/w51//F140M/f140m_nrca3_visit001_vgroup03109_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:36
/orange/adamginsburg/jwst/w51//F140M/f140m_nrca3_visit001_vgroup03109_exp00007_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:35
/orange/adamginsburg/jwst/w51//F140M/f140m_nrca3_visit001_vgroup03109_exp00008_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:36
/orange/adamginsburg/jwst/w51//F140M/f140m_nrca2_visit001_vgroup03109_exp00003_daophot_combined_with

Set DATE-AVG to '2025-05-06T16:44:25.241' from MJD-AVG.
Set DATE-END to '2025-05-06T16:46:28.714' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.273375 from OBSGEO-[XYZ].
Set OBSGEO-H to 1611468465.919 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2025-05-06T16:49:58.105' from MJD-AVG.
Set DATE-END to '2025-05-06T16:52:01.578' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.275157 from OBSGEO-[XYZ].
Set OBSGEO-H to 1611500571.608 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2025-05-06T16:54:37.209' from MJD-AVG.
Set DATE-END to '2025-05-06T16:56:40.682' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.276651 from OBSGEO-[XYZ].
Set OBSGEO-H to 1611527489.341 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2025-05-06T16:39:46.074' from MJD-AVG.
Set DATE-END to '2025-05-06T16:41:49.547' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.271881 from OBSGEO-[XYZ].
Set OBSGEO-H to 1611441536.798 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE

Number of sources in merged table: 106514
Number of sources in merged table with nmatch >= [8 8 8 ... 4 4 4]: 20857
Number of sources in merged table with nmatch >= [8 8 8 ... 4 4 4]: 20846


Number of sources in merged table: 106514
Number of sources in merged table with nmatch >= [8 8 8 ... 4 4 4]: 37007
Number of sources in merged table with nmatch >= [7 7 7 ... 3 3 3]: 36998
/orange/adamginsburg/jwst/w51//F140M/f140m_nrcb2_visit001_vgroup03109_exp00005_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:34
/orange/adamginsburg/jwst/w51//F140M/f140m_nrcb1_visit001_vgroup03109_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:28
/orange/adamginsburg/jwst/w51//F140M/f140m_nrcb1_visit001_vgroup03109_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:29
/orange/adamginsburg/jwst/w51//F140M/f140m_nrcb4_visit001_vgroup03109_exp00005_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:44
/orange/adamginsburg/jwst/w51//F140M/f140m_nrcb3_visit001_vgroup03109_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:38
/orange/adamginsburg/jwst/w51//F140M/f140m_nrcb4_visit001_vg

Set DATE-AVG to '2025-05-06T16:39:46.010' from MJD-AVG.
Set DATE-END to '2025-05-06T16:41:49.483' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.271880 from OBSGEO-[XYZ].
Set OBSGEO-H to 1611441530.624 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2025-05-06T17:03:44.728' from MJD-AVG.
Set DATE-END to '2025-05-06T17:05:48.201' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.279581 from OBSGEO-[XYZ].
Set OBSGEO-H to 1611580286.812 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/astropy/wcs/wcsapi/fitswcs.py:367: UserWarning: 'WCS.all_world2pix' failed to converge to the requested accuracy.
After 20 iterations, the solution is diverging at least for one input point.
  warnings.warn(str(e))
Set DATE-AVG to '2025-05-06T17:19:18.871' from MJD-AVG.
Set DATE-END to '2025-05-06T17:21:22.344' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.284580 from OBSGEO-[XYZ].
Set OBSGEO-H to 161167034

...

Number of sources in merged table: 213111
Number of sources in merged table with nmatch >= [7 7 2 ... 4 4 4]: 19293
Number of sources in merged table with nmatch >= [7 7 2 ... 4 4 4]: 19278
Number of sources in merged table: 213111
Number of sources in merged table with nmatch >= [7 7 2 ... 4 4 4]: 37497
Number of sources in merged table with nmatch >= [6 6 1 ... 3 3 3]: 37485
/orange/adamginsburg/jwst/w51//F360M/f360m_nrcalong_visit001_vgroup03105_exp00006_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:27
/orange/adamginsburg/jwst/w51//F360M/f360m_nrcalong_visit001_vgroup03105_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:27
/orange/adamginsburg/jwst/w51//F360M/f360m_nrcalong_visit001_vgroup03105_exp00007_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:27
/orange/adamginsburg/jwst/w51//F360M/f360m_nrcalong_visit001_vgroup03105_exp00008_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:28
/orange/

Set DATE-AVG to '2025-05-06T16:14:32.219' from MJD-AVG.
Set DATE-END to '2025-05-06T16:16:35.692' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.263774 from OBSGEO-[XYZ].
Set OBSGEO-H to 1611295464.023 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Number of sources in merged table: 121520
Number of sources in merged table with nmatch >= [4 4 2 ... 4 4 4]: 18697
Number of sources in merged table with nmatch >= [4 4 2 ... 4 4 4]: 18688
Number of sources in merged table: 121520
Number of sources in merged table with nmatch >= [4 4 2 ... 4 4 4]: 30200
Number of sources in merged table with nmatch >= [3 3 1 ... 3 3 3]: 30195
/orange/adamginsburg/jwst/w51//F410M/f410m_nrcblong_visit001_vgroup03107_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:29
/orange/adamginsburg/jwst/w51//F410M/f410m_nrcblong_visit001_vgroup03107_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:32
/orange/adamginsburg/jwst/w51//F410M/f410m_nrcblong_visit001_vgroup03107_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:31
/orange/adamginsburg/jwst/w51//F410M/f410m_nrcblong_visit001_vgroup03107_exp00006_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:30
/orange/

Set DATE-AVG to '2025-05-06T17:03:44.856' from MJD-AVG.
Set DATE-END to '2025-05-06T17:05:48.329' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.279582 from OBSGEO-[XYZ].
Set OBSGEO-H to 1611580299.154 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Number of sources in merged table: 75476
Number of sources in merged table with nmatch >= [4 2 4 ... 2 4 2]: 13965
Number of sources in merged table with nmatch >= [4 2 4 ... 2 4 2]: 13960
Number of sources in merged table: 75476
Number of sources in merged table with nmatch >= [4 2 4 ... 2 4 2]: 21789
Number of sources in merged table with nmatch >= [3 1 3 ... 1 3 1]: 21785
/orange/adamginsburg/jwst/w51//F480M/f480m_nrcblong_visit001_vgroup03109_exp00004_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:30
/orange/adamginsburg/jwst/w51//F480M/f480m_nrcblong_visit001_vgroup03109_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:30
/orange/adamginsburg/jwst/w51//F480M/f480m_nrcblong_visit001_vgroup03109_exp00007_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:29
/orange/adamginsburg/jwst/w51//F480M/f480m_nrcblong_visit001_vgroup03109_exp00006_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:31
/orange/ad